In [4]:
pip install pandas numpy scikit-learn


  Using cached pandas-3.0.2-cp312-cp312-manylinux_2_24_x86_64.manylinux_2_28_x86_64.whl.metadata (79 kB)
Using cached pandas-3.0.2-cp312-cp312-manylinux_2_24_x86_64.manylinux_2_28_x86_64.whl (10.9 MB)
Note: you may need to restart the kernel to use updated packages.


In [2]:
import numpy as np
import pandas as pd
import random
import os

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import SVC
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

# -----------------------------
# DEBUG: Show working directory
# -----------------------------
print("Current Directory:", os.getcwd())
print("Files:", os.listdir())

# -----------------------------
# 1. Load Dataset
# -----------------------------
df = pd.read_csv("data.csv", encoding="utf-8")

df = df.rename(columns=lambda x: x.strip().lower())

if "text" not in df.columns or "label" not in df.columns:
    raise ValueError("CSV must contain 'text' and 'label' columns")

texts = df["text"].astype(str).values
labels = df["label"].values

# -----------------------------
# 2. Feature Extraction
# -----------------------------
vectorizer = TfidfVectorizer(max_features=3000)
X = vectorizer.fit_transform(texts)

# -----------------------------
# 3. Train SVM
# -----------------------------
svm = SVC(kernel="linear", probability=True)
svm.fit(X, labels)

# -----------------------------
# 4. Extract Support Vectors
# -----------------------------
sv_indices = svm.support_
SV_texts = texts[sv_indices]
SV_labels = labels[sv_indices]

print(f"\nTotal Samples: {len(texts)}")
print(f"Support Vectors: {len(SV_texts)}")

# -----------------------------
# 5. SG-SV (Stable SV)
# -----------------------------
def get_stable_sv(X, y, runs=5):
    freq = np.zeros(X.shape[0])

    for _ in range(runs):
        idx = np.random.choice(X.shape[0], X.shape[0], replace=True)
        model = SVC(kernel="linear")
        model.fit(X[idx], y[idx])
        freq[model.support_] += 1

    stable = np.where(freq >= (runs // 2))[0]
    return stable

stable_indices = get_stable_sv(X, labels)

print(f"Stable SV count: {len(stable_indices)}")

# -----------------------------
# 6. NASP (Noise Removal)
# -----------------------------
probs = svm.predict_proba(X[sv_indices])
confidence = np.max(probs, axis=1)

clean_mask = confidence > 0.3  # relaxed threshold

if np.sum(clean_mask) == 0:
    print("⚠️ No clean SV found → using all SV")
    clean_SV_texts = SV_texts
    clean_SV_labels = SV_labels
else:
    clean_SV_texts = SV_texts[clean_mask]
    clean_SV_labels = SV_labels[clean_mask]

print(f"Clean SV count: {len(clean_SV_texts)}")

# -----------------------------
# 7. AMA (Augmentation)
# -----------------------------
def augment_text(text):
    words = text.split()
    if len(words) > 1:
        i = random.randint(0, len(words) - 1)
        words[i] = words[i] + "!"
    return " ".join(words)

aug_texts = [augment_text(t) for t in clean_SV_texts]
aug_labels = clean_SV_labels.copy()

# -----------------------------
# 8. Smart Sampling (SAFE)
# -----------------------------
non_sv_mask = np.ones(len(texts), dtype=bool)
non_sv_mask[sv_indices] = False

non_sv_texts = texts[non_sv_mask]
non_sv_labels = labels[non_sv_mask]

if len(non_sv_texts) > 0:
    sample_size = max(1, int(len(non_sv_texts) * 0.3))
    sample_idx = np.random.choice(len(non_sv_texts), sample_size, replace=False)

    sampled_texts = non_sv_texts[sample_idx]
    sampled_labels = non_sv_labels[sample_idx]

    print(f"Sampled Non-SV: {len(sampled_texts)}")
else:
    print("⚠️ No non-SV data available")
    sampled_texts = []
    sampled_labels = []

# -----------------------------
# 9. Merge Dataset
# -----------------------------
final_texts = list(clean_SV_texts) + list(aug_texts) + list(sampled_texts)
final_labels = list(clean_SV_labels) + list(aug_labels) + list(sampled_labels)

print(f"Final Dataset Size: {len(final_texts)}")

# -----------------------------
# 10. Train Models
# -----------------------------
vectorizer_final = TfidfVectorizer(max_features=3000)
X_final = vectorizer_final.fit_transform(final_texts)

X_train, X_test, y_train, y_test = train_test_split(
    X_final, final_labels, test_size=0.2, random_state=42
)

models = {
    "Logistic Regression": LogisticRegression(max_iter=200),
    "Random Forest": RandomForestClassifier()
}

print("\n--- Reduced Dataset Results ---")
for name, model in models.items():
    model.fit(X_train, y_train)
    preds = model.predict(X_test)
    acc = accuracy_score(y_test, preds)
    print(f"{name}: {acc:.4f}")

# -----------------------------
# 11. Baseline (Full Data)
# -----------------------------
X_full_train, X_full_test, y_full_train, y_full_test = train_test_split(
    X, labels, test_size=0.2, random_state=42
)

baseline_model = LogisticRegression(max_iter=200)
baseline_model.fit(X_full_train, y_full_train)
baseline_preds = baseline_model.predict(X_full_test)

baseline_acc = accuracy_score(y_full_test, baseline_preds)

print("\n--- Baseline (Full Data) ---")
print(f"Baseline Accuracy: {baseline_acc:.4f}")

Current Directory: /home/arif/R&D/Paper/SVM
Files: ['test.py', 'venv', 'data.csv', 'helloworld.py', 'requirements.txt', 'reduced_covertype.npz', 'qwen.py', 'app_deepseek.py', 'models.zip', 'app_grok.ipynb', 'helloworld1.py', 'app_gemmini.py', '.ipynb_checkpoints', 'app_kimi (Copy).py', 'app_kimi.py', 'deepseek.py', 'chatgpt.py', 'app_grok.py', 'datasets', 'app_chatgpt.ipynb']

Total Samples: 8
Support Vectors: 8
Stable SV count: 8
Clean SV count: 8
⚠️ No non-SV data available
Final Dataset Size: 16

--- Reduced Dataset Results ---
Logistic Regression: 0.7500
Random Forest: 0.7500

--- Baseline (Full Data) ---
Baseline Accuracy: 0.0000
